# Build EMOPAIR Dataset

Using the results from `validate_LLMs.ipynb`, we can now build the EMOPAIR dataset.

Based on the Alt-Test results, we will use the following LLMs as annotators:
- Valence: No LLMs passed the Alt-Test for valence, so we will not use any LLMs as annotators for this dimension.
- Arousal: GPT-5.1
- Dominance: GPT-5-nano

For Valence, we will use the prior EMOBANK data. 

In [5]:
import os
import pandas as pd 
import pairadigm as pdm

import dotenv

dotenv.load_dotenv(dotenv_path='/Users/mlchrzan/Library/CloudStorage/OneDrive-Personal/Professional/Data Science/Maryland/bechmark analysis pipeline/benchmark-analysis/.env')

True

In [ ]:
# Load EMOBANK data to get the original text data
emobank = pd.read_csv('Data/emobank.csv')
emobank

,id,split,V,A,D,text
0,110CYL068_1036_1079,train,3.00,3.00,3.20,"Remember what she said in my last letter? """
1,110CYL068_1079_1110,test,2.80,3.10,2.80,If I wasn't working here.
2,110CYL068_1127_1130,train,3.00,3.00,3.00,".."""
3,110CYL068_1137_1188,train,3.44,3.00,3.22,Goodwill helps people get off of public assist...
4,110CYL068_1189_1328,train,3.55,3.27,3.46,Sherry learned through our Future Works class ...
...,...,...,...,...,...,...
10057,wwf12_4531_4624,train,3.00,3.50,3.00,Please let it be a constant reminder of all yo...
10058,wwf12_501_591,train,3.80,3.40,3.60,That’s why I want to extend my appreciation fo...
10059,wwf12_592_691,train,3.00,3.00,3.10,And why I’m writing you today to ask you to re...
10060,wwf12_702_921,train,3.33,3.44,3.44,"In fact, I want to urge you to strengthen your..."


In [3]:
# Load CGCOT Prompts
with open('Data/cgcot_prompts/valence.txt', 'r') as file:
    val_prompts = file.read().splitlines()

with open('Data/cgcot_prompts/arousal.txt', 'r') as file:
    aro_prompts = file.read().splitlines()

with open('Data/cgcot_prompts/dominance.txt', 'r') as file:
    dom_prompts = file.read().splitlines()

In [9]:
emopair_aro = pdm.Pairadigm(
    data = emobank, 
    item_id_name='id', 
    text_name='text',
    cgcot_prompts=aro_prompts,
    model_name='gpt-5.1', 
    api_key=os.getenv('OPENAI_API_KEY'),
    base_url="https://us.api.openai.com/v1", 
    target_concept='arousal'
)

emopair_dom = pdm.Pairadigm(
    data = emobank, 
    item_id_name='id', 
    text_name='text',
    cgcot_prompts=dom_prompts,
    model_name='gpt-5-nano', 
    api_key=os.getenv('OPENAI_API_KEY'),
    target_concept='dominance'
)

In [ ]:
emopair_aro.generate_breakdowns()

In [ ]:
pdm.build_pairadigm(
    emopair_aro, 
    num_pairs_per_item=15, 
    allow_ties=True, 
    normalization_scale='negative-one-to-one'
)

,id,split,V,A,D,text
0,110CYL068_1036_1079,train,3.00,3.00,3.20,"Remember what she said in my last letter? """
1,110CYL068_1079_1110,test,2.80,3.10,2.80,If I wasn't working here.
2,110CYL068_1127_1130,train,3.00,3.00,3.00,".."""
3,110CYL068_1137_1188,train,3.44,3.00,3.22,Goodwill helps people get off of public assist...
4,110CYL068_1189_1328,train,3.55,3.27,3.46,Sherry learned through our Future Works class ...
...,...,...,...,...,...,...
10057,wwf12_4531_4624,train,3.00,3.50,3.00,Please let it be a constant reminder of all yo...
10058,wwf12_501_591,train,3.80,3.40,3.60,That’s why I want to extend my appreciation fo...
10059,wwf12_592_691,train,3.00,3.00,3.10,And why I’m writing you today to ask you to re...
10060,wwf12_702_921,train,3.33,3.44,3.44,"In fact, I want to urge you to strengthen your..."
